# Évolution du comportement d'achat

**Objectif** : Comprendre comment le comportement d'achat varie selon le profil du client (occasionnel, typique, fidèle) et identifier les produits qui génèrent de la récurrence.

**Questions traitées** :
1. Comment évoluent les quantités d'achat selon le segment client ?
2. Quel est le taux de réachat global et par segment ?
3. Quels sont les produits les plus rachetés ?

**Impact business attendu** : Identifier à quel moment les clients deviennent fidèles pour concentrer les actions de rétention sur les commandes critiques.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("..")

from src.data_loader import get_data
from src.features import enrich_dataframe

# Style global sobre (anti-chartjunk)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Chargement et enrichissement
data = get_data(raw_dir="../data/raw", processed_path="../data/processed/merged_orders.parquet")
data = enrich_dataframe(data)
print(f"Dataset : {data.shape[0]:,} lignes, {data.shape[1]} colonnes")
data.head(3)

### Vérification rapide de la segmentation

Avant d'analyser, on vérifie que les 3 segments sont bien répartis et cohérents avec les quartiles observés dans le notebook 01.

In [ ]:
# Distribution des utilisateurs par segment
users = data.groupby("user_id").agg(
    segment=("customer_segment", "first"),
    total_orders=("user_total_orders", "first")
).reset_index()

print("Répartition des utilisateurs par segment :")
print(users["segment"].value_counts().sort_index())
print(f"\nTotal : {len(users):,} utilisateurs")

---

## Q1 — Comment évoluent les quantités d'achat selon le segment client ?

On cherche à savoir si les clients fidèles commandent **plus d'articles par commande** que les occasionnels, et si la taille du panier évolue au fil des commandes.

In [ ]:
# Statistiques descriptives de la taille du panier par segment
# On travaille au niveau commande (pas au niveau produit)
orders_level = data.groupby(["order_id", "customer_segment", "order_number"]).agg(
    cart_size=("product_id", "count")
).reset_index()

orders_level.groupby("customer_segment")["cart_size"].describe().round(1)

In [ ]:
# Visualisation 1 : Box plot — distribution de la taille du panier par segment
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=orders_level,
    x="customer_segment",
    y="cart_size",
    showfliers=False,      # On masque les outliers pour la lisibilité
    ax=ax
)

ax.set_xlabel("Segment client")
ax.set_ylabel("Nombre d'articles par commande")
ax.set_title("Distribution de la taille du panier selon le segment client")

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation 2 : Line plot — évolution de la taille moyenne du panier
# au fil des commandes successives, par segment
#
# On limite à order_number ≤ 20 pour comparer les segments sur une plage commune
# (les occasionnels n'ont par définition pas plus de 6 commandes)

evo = (
    orders_level[orders_level["order_number"] <= 20]
    .groupby(["order_number", "customer_segment"])["cart_size"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))

for segment, group in evo.groupby("customer_segment", observed=True):
    ax.plot(group["order_number"], group["cart_size"], marker="o", markersize=4, label=segment)

ax.set_xlabel("Numéro de commande")
ax.set_ylabel("Taille moyenne du panier")
ax.set_title("Évolution de la taille du panier au fil des commandes")
ax.legend(title="Segment")
ax.set_xticks(range(1, 21))

plt.tight_layout()
plt.show()

### Observation Q1

*À compléter après exécution :*

- Les clients fidèles ont-ils des paniers plus gros ou plus petits ?
- La taille du panier augmente-t-elle avec le numéro de commande ?
- Y a-t-il un plateau ou une tendance continue ?

---

## Q2 — Quel est le taux de réachat global et par segment client ?

La colonne `reordered` vaut 1 si le client a **déjà commandé ce produit avant**, 0 sinon. Un taux élevé signifie que les clients rachètent les mêmes produits — signe d'habitudes et de fidélité produit.

In [ ]:
# Taux de réachat global
global_reorder = data["reordered"].mean()
print(f"Taux de réachat global : {global_reorder:.1%}")

# Par segment
reorder_by_segment = (
    data.groupby("customer_segment", observed=True)["reordered"]
    .mean()
    .reset_index()
    .rename(columns={"reordered": "taux_reachat"})
)
reorder_by_segment["taux_reachat_pct"] = (reorder_by_segment["taux_reachat"] * 100).round(1)
reorder_by_segment

In [ ]:
# Visualisation : Bar chart avec ligne de moyenne globale
fig, ax = plt.subplots(figsize=(7, 5))

colors = sns.color_palette("muted", n_colors=3)
bars = ax.bar(
    reorder_by_segment["customer_segment"].astype(str),
    reorder_by_segment["taux_reachat"] * 100,
    color=colors
)

# Ligne de moyenne globale
ax.axhline(y=global_reorder * 100, color="grey", linestyle="--", linewidth=1)
ax.text(
    2.55, global_reorder * 100 + 0.5,
    f"Moyenne globale : {global_reorder:.1%}",
    color="grey", fontsize=10, ha="right"
)

# Annotations sur les barres
for bar, val in zip(bars, reorder_by_segment["taux_reachat_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        f"{val}%", ha="center", fontsize=11, fontweight="bold"
    )

ax.set_xlabel("Segment client")
ax.set_ylabel("Taux de réachat (%)")
ax.set_title("Taux de réachat par segment client")
ax.set_ylim(0, max(reorder_by_segment["taux_reachat"] * 100) + 10)

plt.tight_layout()
plt.show()

### Observation Q2

*À compléter après exécution :*

- Quel segment a le taux de réachat le plus élevé ?
- L'écart entre occasionnel et fidèle est-il important ?
- Attention : les premières commandes (order_number=1) ont par définition reordered=0, ce qui tire le taux des occasionnels vers le bas mécaniquement.

---

## Q3 — Quels sont les produits les plus rachetés ?

On identifie les produits qui génèrent le plus de récurrence : ceux que les clients rachètent commande après commande. Ce sont les produits "piliers" du panier.

In [ ]:
# Top 20 des produits les plus réachetés (en volume)
top_reordered = (
    data[data["reordered"] == 1]
    .groupby("product_name")
    .size()
    .nlargest(20)
    .reset_index(name="nb_reorders")
)
top_reordered

In [ ]:
# Visualisation : Bar chart horizontal
fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    top_reordered["product_name"][::-1],   # Inverser pour que le #1 soit en haut
    top_reordered["nb_reorders"][::-1],
    color=sns.color_palette("muted")[0]
)

ax.set_xlabel("Nombre de réachats")
ax.set_title("Top 20 des produits les plus rachetés")

plt.tight_layout()
plt.show()

In [ ]:
# Complément : taux de réachat PAR produit (parmi les produits vendus > 1000 fois)
# Un produit peut être très vendu mais peu racheté, ou peu vendu mais toujours racheté.

product_stats = (
    data.groupby("product_name")
    .agg(
        total_ventes=("reordered", "count"),
        nb_reorders=("reordered", "sum"),
    )
)
product_stats["taux_reachat"] = product_stats["nb_reorders"] / product_stats["total_ventes"]

# Filtrer les produits avec assez de volume pour être significatifs
popular = product_stats[product_stats["total_ventes"] >= 1000].sort_values("taux_reachat", ascending=False)

print("Top 15 — Taux de réachat le plus élevé (min 1000 ventes) :")
popular.head(15).round(2)

In [ ]:
# Visualisation : Scatter — volume de ventes vs taux de réachat
# Chaque point = un produit. On cherche ceux en haut à droite (gros volume + fort réachat)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    product_stats["total_ventes"],
    product_stats["taux_reachat"] * 100,
    alpha=0.15, s=10, color=sns.color_palette("muted")[0]
)

# Annoter les top produits
for _, row in popular.head(5).iterrows():
    ax.annotate(
        row.name,
        (row["total_ventes"], row["taux_reachat"] * 100),
        fontsize=9, ha="left",
        arrowprops=dict(arrowstyle="-", color="grey", lw=0.5),
        xytext=(10, 5), textcoords="offset points"
    )

ax.set_xlabel("Volume total de ventes")
ax.set_ylabel("Taux de réachat (%)")
ax.set_title("Volume vs Taux de réachat par produit")

plt.tight_layout()
plt.show()

### Observation Q3

*À compléter après exécution :*

- Les produits les plus rachetés sont-ils des produits frais du quotidien (fruits, lait, œufs) ?
- Y a-t-il une corrélation entre volume de ventes et taux de réachat ?
- Les produits bio apparaissent-ils souvent dans le top ?

---

## Synthèse et pistes business

*À compléter après toutes les analyses :*

- **Constat principal** : ...
- **Action 1** : ...
- **Action 2** : ...
- **Action 3** : ...